# 3 · Train, evaluate and export the models

Uses the search result from notebook 2 (`data/models/hyperparameters.json`, or defaults if it does not exist yet) and

1. evaluates each model **leave-one-sausage-out**: train on four sausages, score on the fifth;
2. saves one model per fold (`data/models/folds/`). The demonstrator replays a sausage with the model that never saw it;
3. trains the final models on all annotated sausages (`data/models/final/`);
4. writes `manifest.json` and `metrics.json`, which the demonstrator's evaluation panel shows.

It also checks the models on the **unannotated series 2** (summer → spring): phases can only advance in time, so the
predicted phase should rise there too. A model that learned the season instead of the ripening runs backwards
(details and the comparison of feature sets: notebook 04).

`FEATURE_SET` must match the one used in notebook 2. `LIMIT_SIZE` caps the Random Forest (100 trees, depth 10). An unlimited forest on this data grows to hundreds of MB.
**Train with the same scikit-learn version the demo laptop runs**, because model files are not portable across versions.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import glob
import matplotlib.pyplot as plt
import pandas as pd
from ml_core import data
from ml_core.training import (DEFAULT_AUGMENTATION, DEFAULT_CONTEXT, DEFAULT_FEATURE_SET, DEFAULT_STRIDE,
                              DEFAULT_WINDOW_SIZE, DEMO_MODELS, PROGRESSION_UNTIL, load_windows, metrics_table,
                              train_and_evaluate)

In [ ]:
CONVERTED_DIR = data.CONVERTED_DIR
MODELS_DIR = data.MODELS_DIR
WINDOW_SIZE, STRIDE = DEFAULT_WINDOW_SIZE, DEFAULT_STRIDE   # uplinks per window, step between windows
CONTEXT = DEFAULT_CONTEXT           # long-term context horizons, e.g. ("24h", "7d"); compared in notebook 05
FEATURE_SET = DEFAULT_FEATURE_SET   # see FEATURE_SETS in ml_core/training.py
AUGMENT = DEFAULT_AUGMENTATION      # training-only augmentation (names: AUGMENTATIONS in ml_core/augmentation.py,
                                    # compared in notebook 04); [] = none
MODELS = list(DEMO_MODELS)          # model families to export; the dashboards show all exported models
LIMIT_SIZE = True
FOLD_MODELS = True
QUICK = False
N_JOBS = -1          # worker processes for folds (-1 = all cores); lower it if RAM runs short

In [ ]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE, context=CONTEXT)
unannotated = None
if glob.glob(os.path.join(CONVERTED_DIR, "series_2_*.csv")):
    unannotated = load_windows(CONVERTED_DIR, "series_2_*.csv", WINDOW_SIZE, STRIDE, annotated=False,
                               classes=windows.classes, context=CONTEXT)
windows.summary()

In [ ]:
metrics = train_and_evaluate(windows, MODELS_DIR, FEATURE_SET, unannotated, augmentation=AUGMENT, keys=MODELS,
                             limit_size=LIMIT_SIZE, fold_models=FOLD_MODELS, quick=QUICK, n_jobs=N_JOBS)

## Results (mean ± std over the held-out sausages)

In [ ]:
metrics_table(metrics)

In [ ]:
# balanced accuracy per held-out sausage
pd.DataFrame({m["name"]: {f["held_out"]: round(f["balanced_accuracy"], 3) for f in m["folds"]}
              for m in metrics["models"].values()})

In [ ]:
# confusion matrices, row-normalised (rows: annotation, columns: prediction)
for m in metrics["models"].values():
    cm = pd.DataFrame(m["confusion_matrix"], index=metrics["classes"], columns=metrics["classes"])
    print(m["name"])
    display((cm.div(cm.sum(axis=1), axis=0) * 100).round(1))

In [ ]:
# how often two models give the same label on held-out windows
pd.Series(metrics["agreement_on_held_out"]).round(3)

## Plausibility on the unannotated series 2

Predicted phase over time (smoothed over one day). Plausible: a staircase going up. A model that follows the
temperature instead goes down in autumn and winter. ρ in the table above: +1 = phases only advance.

In [ ]:
# the exported final models (fitted on all annotated sausages), as scored in the table above
if metrics["progressions"]:
    classes = metrics["classes"]
    fig, axes = plt.subplots(1, len(metrics["progressions"]), figsize=(15, 3.2), sharey=True, squeeze=False)
    for ax, (key, progression) in zip(axes[0], metrics["progressions"].items()):
        for rec, s in progression.items():
            ax.plot(s.index, s.values, label=rec, lw=1)
        ax.axvline(pd.Timestamp(PROGRESSION_UNTIL), color="grey", ls="--", lw=1)
        ax.set(title=metrics["models"][key]["name"], yticks=range(len(classes)), yticklabels=classes)
        ax.tick_params(axis="x", rotation=30)
    axes[0][0].legend(fontsize=7)
    plt.tight_layout()
    plt.show()

**Next:** start the demonstrator (`scripts/start_demo.ps1`). It loads the models from `data/models/`.
Read the scores with care: all annotated sausages come from one curing run.